# Notebook 01 — Data Generation & Intro to pandas

**Goal**: Generate our synthetic banking dataset and learn the basics of `pandas` — the most important library for ML data work.

**What you'll learn:**
- What a DataFrame is and how to work with it
- How to load, inspect, and slice data
- How to check for missing values
- How to do basic statistics

---

## Step 1 — Generate the data

We use our custom generator to create 1000 fake bank customers.
Each customer has financial attributes: income, spending, savings, credit score, etc.

The generator secretly assigns each customer to one of 4 archetypes:
- **Young Spender** — low income, high spending, low savings
- **Stable Saver** — mid income, low spending, high savings
- **High Earner** — high income, high savings, excellent credit
- **Debt Stressed** — low income, high loans, poor credit

Our ML model doesn't know these labels — it has to *discover* them from the data.

In [ ]:
import sys
sys.path.append('..')  # so we can import from src/

import os
import pandas as pd
import numpy as np

from src.data_generator import generate_customers

# Generate 1000 customers
df = generate_customers(n_customers=1000, seed=42)

# Save to disk so other notebooks can load it
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/customers_raw.csv', index=False)

print('Data saved to data/customers_raw.csv')
print(f'Shape: {df.shape}')  # (rows, columns)

## Step 2 — The DataFrame: your main data structure

A **DataFrame** is a table — rows are observations (customers), columns are features (attributes).
Think of it as a smarter spreadsheet you can program.

`df.head()` shows the first 5 rows.

In [ ]:
df.head()

In [ ]:
# Shape: (rows, columns)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print()

# Column names
print('Columns:', df.columns.tolist())

## Step 3 — Data types

Every column has a data type. ML models only understand numbers, so we need to know which columns are:
- `int64` / `float64` → numeric (ready to use)
- `object` → text/categorical (needs encoding later)

`df.info()` gives a quick overview.

In [ ]:
df.info()

## Step 4 — Summary statistics

`df.describe()` gives count, mean, min, max, and percentiles for every numeric column.
This is your first "feel" for the data.

In [ ]:
df.describe().round(2)

**What to look for:**
- Large gap between mean and max → possible outliers
- min = 0 on `loan_amount` → some customers have no loan (expected)
- Different scales (income is in 100,000s, age is in 20-60) → we'll need to **scale** in notebook 03

## Step 5 — Selecting columns

Two ways to pick columns:
- `df['column_name']` → one column (returns a Series)
- `df[['col1', 'col2']]` → multiple columns (returns a DataFrame)

In [ ]:
# Single column — returns a Series (like a list with an index)
df['annual_income'].head()

In [ ]:
# Multiple columns — returns a DataFrame
df[['customer_id', 'annual_income', 'credit_score']].head()

## Step 6 — Filtering rows

Use conditions inside `[]` to filter, just like a WHERE clause in SQL.

In [ ]:
# Customers with annual income > 100,000
high_earners = df[df['annual_income'] > 100_000]
print(f'High earners: {len(high_earners)} customers')
high_earners.head()

In [ ]:
# Multiple conditions: use & (and), | (or)
# Young customers with poor credit
risky_young = df[(df['age'] < 30) & (df['credit_score'] < 600)]
print(f'Young + poor credit: {len(risky_young)} customers')

## Step 7 — Missing values

Real data always has missing values. We introduced ~3% missing in `savings_balance` and `loan_amount`.

`df.isnull().sum()` counts missing values per column.

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print()

# As a percentage
print('Missing %:')
print((df.isnull().sum() / len(df) * 100).round(2))

We'll handle these missing values properly in **notebook 03 (Preprocessing)**.

## Step 8 — Groupby: aggregate by category

`groupby` groups rows by a column value, then applies an aggregation.
Think of it as: "for each archetype, what's the average income?"

In [ ]:
# Average income and savings per archetype
# (We're peeking at archetypes here just to understand the data — the ML model won't see this)
df.groupby('archetype')[['annual_income', 'savings_balance', 'credit_score', 'loan_amount']].mean().round(0)

In [ ]:
# How many customers per archetype?
df['archetype'].value_counts()

## Step 9 — Adding a computed column

Feature engineering starts here — creating new columns from existing ones.

In [ ]:
# Spending ratio: monthly spending as % of monthly income
df['spending_ratio'] = (df['monthly_spending'] / (df['annual_income'] / 12)).round(4)

# Savings ratio: savings as % of annual income
df['savings_ratio'] = (df['savings_balance'] / df['annual_income']).round(4)

print('New columns added: spending_ratio, savings_ratio')
df[['annual_income', 'monthly_spending', 'spending_ratio', 'savings_balance', 'savings_ratio']].head()

## Checkpoint — What you learned

| Concept | pandas method |
|---|---|
| Load/view data | `df.head()`, `df.shape` |
| Data types | `df.info()` |
| Summary stats | `df.describe()` |
| Select columns | `df['col']`, `df[['a','b']]` |
| Filter rows | `df[condition]` |
| Missing values | `df.isnull().sum()` |
| Group & aggregate | `df.groupby().mean()` |
| New column | `df['new'] = expression` |

**Next up: Notebook 02 — Exploratory Data Analysis (EDA)**
We'll visualize distributions, spot outliers, and understand feature correlations.